# Set up server

In [ ]:
from chi import context, lease, server
context.version = "1.0"
context.choose_site(default="CHI@TACC")
context.choose_project()

In [ ]:
from chi import hardware

node_type = "compute_cascadelake"
available_nodes = hardware.get_nodes(node_type=node_type, filter_reserved=True)
if available_nodes:
    print(f"There currently are {len(available_nodes)} {node_type} nodes ready to use")
    hardware.show_nodes(available_nodes)
else:
    print(f"All {node_type} nodes are in use! You could use next_free_timeslot to see how long you need to wait, or use the calendar.")

In [ ]:
from datetime import timedelta
import os

my_lease = lease.Lease(f"{os.getenv('USER')}-mu-slope", duration=timedelta(hours=3))
my_lease.add_node_reservation(nodes=[available_nodes[0]]) # or you could use node_type=node_type
my_lease.add_fip_reservation(1) # include a floating ip
my_lease.submit(idempotent=True)

In [ ]:
my_server = server.Server(
    f"{os.getenv('USER')}-mu-slope",
    reservation_id=my_lease.node_reservations[0]["id"],
    image_name="CC-Ubuntu22.04", # or use image_name
)
my_server.submit(idempotent=True)

In [ ]:
fip = my_lease.get_reserved_floating_ips()[0]
my_server.associate_floating_ip(fip)

# Build synthetic logs

In [ ]:
from pathlib import Path
import os
import json
import random
import shutil
import gzip
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
CLP_OUT_DIR = OUT_DIR / "clp"

DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
CLP_OUT_DIR.mkdir(exist_ok=True)

RAW_PATH = DATA_DIR / "synthetic_logs.jsonl"

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
random.seed(0)

services = ["auth", "payment", "search", "feed", "profile"]
levels = ["INFO", "WARN", "ERROR"]
paths = ["/login", "/logout", "/checkout", "/search", "/profile", "/feed"]
methods = ["GET", "POST"]
regions = ["us-east", "us-west", "eu-central"]

def make_record(i: int) -> dict:
    service = random.choice(services)
    level = random.choices(levels, weights=[0.80, 0.15, 0.05])[0]

    record = {
        "timestamp": f"2024-01-01T00:{(i // 60) % 60:02d}:{i % 60:02d}.000Z",
        "level": level,
        "service": service,
        "host": f"{service}-{random.randint(1, 20)}",
        "trace_id": f"trace-{random.randint(1, 5000)}",
        "request": {
            "method": random.choice(methods),
            "path": random.choice(paths),
            "region": random.choice(regions),
        },
        "latency_ms": random.randint(1, 500),
        "message": random.choice([
            "request completed",
            "request retried",
            "dependency timeout",
            "cache miss",
            "cache hit",
            "invalid user token",
        ]),
    }

    # Controlled schema variation by service.
    if service == "payment":
        record["payment"] = {
            "currency": random.choice(["USD", "EUR", "GBP"]),
            "amount_bucket": random.choice(["small", "medium", "large"]),
            "processor": random.choice(["stripe", "adyen", "internal"]),
        }
    elif service == "search":
        record["search"] = {
            "query_type": random.choice(["keyword", "semantic", "autocomplete"]),
            "num_results": random.randint(0, 100),
        }
    elif service == "auth":
        record["auth"] = {
            "provider": random.choice(["password", "google", "github"]),
            "success": level != "ERROR",
        }

    return record

def write_dataset(n_records: int, path: Path) -> None:
    with path.open("w") as f:
        for i in range(n_records):
            f.write(json.dumps(make_record(i), separators=(",", ":")) + "\n")

N = 100_000
write_dataset(N, RAW_PATH)

raw_size = RAW_PATH.stat().st_size
print(f"records = {N:,}")
print(f"raw size = {raw_size / 1e6:.2f} MB")

In [ ]:
def file_size(path: Path) -> int:
    return Path(path).stat().st_size

def dir_size(path: Path) -> int:
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def run(cmd, *, cwd=None, check=True):
    print("+", " ".join(map(str, cmd)))
    out = subprocess.run(
        list(map(str, cmd)),
        cwd=cwd,
        check=check,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(out.stdout[-2000:])
    return out

# Compress with baselines

In [ ]:
GZIP_PATH = OUT_DIR / "synthetic_logs.jsonl.gz"

with RAW_PATH.open("rb") as f_in, gzip.open(GZIP_PATH, "wb", compresslevel=9) as f_out:
    shutil.copyfileobj(f_in, f_out)

print("gzip size MB:", file_size(GZIP_PATH) / 1e6)

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zstandard"], check=True)
import zstandard as zstd

def zstd_compress(input_path: Path, output_path: Path, level: int):
    cctx = zstd.ZstdCompressor(level=level)
    with input_path.open("rb") as f_in, output_path.open("wb") as f_out:
        cctx.copy_stream(f_in, f_out)

ZSTD3_PATH = OUT_DIR / "synthetic_logs.jsonl.zst3"
ZSTD19_PATH = OUT_DIR / "synthetic_logs.jsonl.zst19"

zstd_compress(RAW_PATH, ZSTD3_PATH, level=3)
zstd_compress(RAW_PATH, ZSTD19_PATH, level=19)

print("zstd -3 size MB:", file_size(ZSTD3_PATH) / 1e6)
print("zstd -19 size MB:", file_size(ZSTD19_PATH) / 1e6)

# Download mu slope (part of CLP)

In [ ]:
from pathlib import Path
import subprocess
import shutil
import os

BIN_DIR = ROOT / "bin"
BIN_DIR.mkdir(exist_ok=True)

DEB_PATH = ROOT / "clp-core_0.12.0-1_amd64.deb"
EXTRACT_DIR = ROOT / "clp_core_extracted"

# Option A: download directly from GitHub release.
# If this fails because your environment has no internet, manually upload the .deb
# into the same directory as the notebook and rerun this cell.
if not DEB_PATH.exists():
    subprocess.run([
        "wget",
        "-O", str(DEB_PATH),
        "https://github.com/y-scope/clp/releases/download/v0.12.0/clp-core_0.12.0-1_amd64.deb"
    ], check=True)

# Extract .deb without sudo.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

subprocess.run([
    "dpkg-deb", "-x",
    str(DEB_PATH),
    str(EXTRACT_DIR)
], check=True)

# Find clp-s inside the extracted package.
matches = list(EXTRACT_DIR.rglob("clp-s"))
print(matches)

assert matches, "Could not find clp-s in the extracted CLP core package."

CLP_S_SRC = matches[0]
CLP_S = BIN_DIR / "clp-s"

shutil.copy2(CLP_S_SRC, CLP_S)
subprocess.run(["chmod", "+x", str(CLP_S)], check=True)

CLP_S = CLP_S.resolve()

# Sanity check.
# subprocess.run([str(CLP_S), "--help"], check=True)

print("Using CLP_S =", CLP_S)

In [ ]:
LIB_DIR = ROOT / "local_libs"
LIB_DIR.mkdir(exist_ok=True)

OPENSSL_DEB = ROOT / "libssl1.1_1.1.1f-1ubuntu2.24_amd64.deb"
OPENSSL_EXTRACT = ROOT / "libssl1_1_extracted"

if not OPENSSL_DEB.exists():
    subprocess.run([
        "wget",
        "-O", str(OPENSSL_DEB),
        "http://security.ubuntu.com/ubuntu/pool/main/o/openssl/libssl1.1_1.1.1f-1ubuntu2.24_amd64.deb"
    ], check=True)

if OPENSSL_EXTRACT.exists():
    shutil.rmtree(OPENSSL_EXTRACT)

subprocess.run([
    "dpkg-deb", "-x",
    str(OPENSSL_DEB),
    str(OPENSSL_EXTRACT)
], check=True)

for libname in ["libssl.so.1.1", "libcrypto.so.1.1"]:
    matches = list(OPENSSL_EXTRACT.rglob(libname))
    print(libname, matches)
    assert matches, f"Could not find {libname}"
    shutil.copy2(matches[0], LIB_DIR / libname)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{LIB_DIR}:{env.get('LD_LIBRARY_PATH', '')}"

subprocess.run([str(CLP_S), "--help"], check=True, env=env)

print("clp-s works with local OpenSSL 1.1 libraries")

# Run mu slope

In [ ]:
CLP_ARCHIVE_DIR = CLP_OUT_DIR / "archives"

if CLP_ARCHIVE_DIR.exists():
    shutil.rmtree(CLP_ARCHIVE_DIR)

CLP_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    str(CLP_S), "c",
    "--timestamp-key", "timestamp",
    "--target-encoded-size", "1073741824",
    "--compression-level", "6",
    str(CLP_ARCHIVE_DIR),
    str(RAW_PATH),
], check=True, env=env)

clp_size = dir_size(CLP_ARCHIVE_DIR)

print(f"CLP archive dir: {CLP_ARCHIVE_DIR}")
print(f"CLP archive size: {clp_size / 1e6:.3f} MB")

In [ ]:
sizes = {
    "raw JSONL": file_size(RAW_PATH),
    "gzip -9": file_size(GZIP_PATH),
    "zstd -3": file_size(ZSTD3_PATH),
    "zstd -19": file_size(ZSTD19_PATH),
    "CLP / μSlope": dir_size(CLP_ARCHIVE_DIR),
}

df = pd.DataFrame([
    {
        "method": method,
        "size_bytes": size,
        "size_MB": size / 1e6,
        "compression_ratio": sizes["raw JSONL"] / size,
    }
    for method, size in sizes.items()
])

df

# Plot comparison

In [ ]:
plot_df = df[df["method"] != "raw JSONL"].copy()

plt.figure(figsize=(8, 4))
plt.bar(plot_df["method"], plot_df["compression_ratio"])
plt.ylabel("Compression ratio = raw JSONL size / compressed size")
plt.xlabel("Method")
plt.title("Reduced μSlope / CLP Compression-Ratio Reproduction")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Decompress

In [ ]:
DECOMP_DIR = CLP_OUT_DIR / "decompressed"

if DECOMP_DIR.exists():
    shutil.rmtree(DECOMP_DIR)

DECOMP_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    str(CLP_S), "x",
    str(CLP_ARCHIVE_DIR),
    str(DECOMP_DIR),
], check=True, env=env)

print("Decompressed files:")
for p in DECOMP_DIR.rglob("*"):
    if p.is_file():
        print(p)

# Figure 13 Reproduction

In [ ]:
import json
import uuid
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zstandard as zstd

FIG13_DIR = ROOT / "fig13_sweep"
FIG13_DATA = FIG13_DIR / "data"
FIG13_OUT = FIG13_DIR / "outputs"

FIG13_DATA.mkdir(parents=True, exist_ok=True)
FIG13_OUT.mkdir(parents=True, exist_ok=True)

# Paper uses 670K records and roughly 1GB per dataset.
# Start smaller for a notebook. Increase to 670_000 if runtime is acceptable.
N_RECORDS = 5_000
N_FIELDS = 20

P_VALUES = [0, 1e-4, 1e-3, 1e-2, 1e-1, 0.5, 1.0]
REPETITION_RATIOS = [1000, 100, 10, 1]

rng = np.random.default_rng(0)

def make_uuid_pool(n, seed):
    local_rng = np.random.default_rng(seed)
    # UUID-looking strings, deterministic and cheaper than uuid.uuid4 in huge loops.
    return [uuid.UUID(int=int(x)).hex for x in local_rng.integers(0, 2**63 - 1, size=n, dtype=np.int64)]

def get_schema(schema_id, schema_cache):
    """
    Each schema is a list of 20 UUID-looking key names.
    """
    if schema_id not in schema_cache:
        # Stable per schema_id
        schema_cache[schema_id] = [f"k_{schema_id}_{j}_{uuid.uuid4().hex}" for j in range(N_FIELDS)]
    return schema_cache[schema_id]

def sample_schema_id(i, P, rng):
    """
    Paper distribution:
        Pr(schema n) = P * (1-P)^n, n starts at 0.

    P=0 is used here to mean the paper's limiting case where every record
    has a unique schema.
    """
    if P == 0:
        return i
    if P == 1:
        return 0
    return int(rng.geometric(P) - 1)

def generate_fig13_dataset(path, P, repetition_ratio, n_records=N_RECORDS, n_fields=N_FIELDS, seed=0):
    """
    Generate JSONL synthetic logs matching Figure 13's construction:
      - every record has 20 fields
      - every key is UUID-like
      - every value is UUID-like
      - schema repetitiveness controlled by P
      - value repetitiveness controlled by repetition_ratio

    repetition_ratio = total variable values / unique variable values
    """
    rng = np.random.default_rng(seed)
    total_values = n_records * n_fields
    n_unique_values = max(1, total_values // repetition_ratio)

    value_pool = make_uuid_pool(n_unique_values, seed + 12345)
    schema_cache = {}

    with path.open("w") as f:
        for i in range(n_records):
            sid = sample_schema_id(i, P, rng)
            keys = get_schema(sid, schema_cache)

            # Draw values uniformly from pool.
            vals = rng.choice(value_pool, size=n_fields, replace=True)

            rec = {keys[j]: str(vals[j]) for j in range(n_fields)}
            f.write(json.dumps(rec, separators=(",", ":")) + "\n")

    return {
        "path": path,
        "P": P,
        "repetition_ratio": repetition_ratio,
        "n_records": n_records,
        "n_fields": n_fields,
        "n_unique_values": n_unique_values,
        "n_schemas_materialized": len(schema_cache),
        "raw_size_bytes": path.stat().st_size,
    }

In [ ]:
def path_size(path: Path) -> int:
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def zstd_compress(input_path: Path, output_path: Path, level: int = 3):
    cctx = zstd.ZstdCompressor(level=level)
    with input_path.open("rb") as f_in, output_path.open("wb") as f_out:
        cctx.copy_stream(f_in, f_out)

def clp_compress_json(input_path: Path, archive_dir: Path):
    if archive_dir.exists():
        shutil.rmtree(archive_dir)
    archive_dir.mkdir(parents=True, exist_ok=True)

    # No timestamp-key here because Figure 13 synthetic records are UUID keys/values.
    subprocess.run([
        str(CLP_S), "c",
        "--target-encoded-size", "1073741824",
        "--compression-level", "3",
        str(archive_dir),
        str(input_path),
    ], check=True, env=env)

    return path_size(archive_dir)

In [ ]:
results = []

for rr in REPETITION_RATIOS:
    for P in P_VALUES:
        label_P = "0" if P == 0 else str(P).replace(".", "p")
        dataset_name = f"synthetic_P={label_P}_rr={rr}.jsonl"
        raw_path = FIG13_DATA / dataset_name

        print(f"\n=== P={P}, repetition_ratio={rr} ===")

        meta = generate_fig13_dataset(
            raw_path,
            P=P,
            repetition_ratio=rr,
            n_records=N_RECORDS,
            n_fields=N_FIELDS,
            seed=hash((P, rr)) % (2**32),
        )

        raw_size = raw_path.stat().st_size

        zstd_path = FIG13_OUT / f"{dataset_name}.zst"
        zstd_compress(raw_path, zstd_path, level=3)
        zstd_size = zstd_path.stat().st_size

        clp_archive_dir = FIG13_OUT / f"{dataset_name}.clp_archive"
        start = time.perf_counter()
        clp_size = clp_compress_json(raw_path, clp_archive_dir)
        clp_time = time.perf_counter() - start

        row_base = {
            "P": P,
            "P_label": "0" if P == 0 else f"{P:g}",
            "repetition_ratio": rr,
            "raw_size_bytes": raw_size,
            "n_schemas_materialized": meta["n_schemas_materialized"],
            "n_unique_values": meta["n_unique_values"],
        }

        results.append({
            **row_base,
            "method": "μSlope",
            "compressed_size_bytes": clp_size,
            "compression_ratio": raw_size / clp_size,
            "compress_time_s": clp_time,
        })

        results.append({
            **row_base,
            "method": "Zstandard",
            "compressed_size_bytes": zstd_size,
            "compression_ratio": raw_size / zstd_size,
            "compress_time_s": None,
        })

        print(f"raw MB: {raw_size / 1e6:.2f}")
        print(f"schemas materialized: {meta['n_schemas_materialized']}")
        print(f"zstd ratio: {raw_size / zstd_size:.2f}")
        print(f"μSlope ratio: {raw_size / clp_size:.2f}")

df13 = pd.DataFrame(results)
df13

In [ ]:
x_labels = ["0", "10$^{-4}$", "10$^{-3}$", "10$^{-2}$", "10$^{-1}$", "0.5", "1"]
P_order = [0, 1e-4, 1e-3, 1e-2, 1e-1, 0.5, 1.0]
x = np.arange(len(P_order))

plt.figure(figsize=(9, 5))

for rr in [1000, 100, 10, 1]:
    y = []
    for P in P_order:
        val = df13[
            (df13["method"] == "μSlope") &
            (df13["repetition_ratio"] == rr) &
            (df13["P"] == P)
        ]["compression_ratio"].iloc[0]
        y.append(val)

    plt.plot(x, y, marker="o", linewidth=2, label=f"μSlope({rr})")

# The paper only plots Zstandard for rr=1000 and rr=1.
for rr, linestyle in [(1000, "--"), (1, ":")]:
    y = []
    for P in P_order:
        val = df13[
            (df13["method"] == "Zstandard") &
            (df13["repetition_ratio"] == rr) &
            (df13["P"] == P)
        ]["compression_ratio"].iloc[0]
        y.append(val)

    plt.plot(x, y, marker="o", linewidth=2, linestyle=linestyle, color="gray", label=f"Zstandard({rr})")

plt.xticks(x, x_labels)
plt.xlabel("schema repetitiveness (P)")
plt.ylabel("compression ratio")
plt.title("Compression ratio of μSlope against Zstandard on synthetic logs")
plt.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.25))
plt.tight_layout()
plt.show()